# TASK 1: PROJECT OVERVIEW & KEY LEARNING OBJECTIVES

# TASK 2: SETTING UP YOUR ENVIRONMENT & BUILD AN AGENT WITH NO MEMORY


In [1]:
import os
import google.generativeai as genai
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Load environment variables and configure client
load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=gemini_api_key)

print("Gemini client configured.")

c:\Users\EduTech\anaconda3\envs\py310\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Gemini client configured.


c:\Users\EduTech\anaconda3\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\hadoop\tmp\ipykernel_28124\3319295661.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
# Define a function for printing markdown cells
def print_markdown(text):
    display(Markdown(text))

In [3]:
# ---------------------------------------------------------------
# Agent class — lightweight wrapper around the Gemini SDK.
# Mirrors the structure of the OpenAI Agents SDK Agent class.
# ---------------------------------------------------------------
class Agent:
    """A simple AI agent backed by a Gemini model."""

    def __init__(self, name: str, instructions: str, model: str = "gemini-2.5-flash-lite"):
        """
        name         – human-readable label for the agent
        instructions – system prompt that defines the agent's behaviour
        model        – Gemini model string (default: gemini-2.5-flash-lite)
        """
        self.name = name
        self.instructions = instructions
        self.model = model
        # Create the underlying Gemini generative model with a system instruction
        self._client = genai.GenerativeModel(
            model_name=model,
            system_instruction=instructions,
        )


# ---------------------------------------------------------------
# RunResult — holds the result of a Runner.run() call.
# ---------------------------------------------------------------
class RunResult:
    """Holds the result of a Runner.run() call."""

    def __init__(self, final_output: str, usage=None):
        self.final_output = final_output
        self.usage = usage  # token usage metadata (if available)


# ---------------------------------------------------------------
# Runner class — executes the agent and returns a RunResult.
# Supports an optional `session` for memory (conversation history).
# ---------------------------------------------------------------
class Runner:
    """Runs an Agent against a user input and returns a RunResult."""

    @staticmethod
    def run(starting_agent: Agent, input: str, session=None) -> RunResult:
        """
        starting_agent – the Agent to use
        input          – the user message / question to process
        session        – optional session object for persistent memory
                         (must implement get_history() and add_turn())
        """
        if session is None:
            # ── NO MEMORY: single-turn, stateless call ──────────────
            response = starting_agent._client.generate_content(input)
        else:
            # ── WITH MEMORY: multi-turn, stateful call ──────────────
            # Build the full conversation history for this request
            history = session.get_history()

            # Start a Gemini chat session with the existing history
            chat = starting_agent._client.start_chat(history=history)

            # Send the new user message
            response = chat.send_message(input)

            # Persist both the user turn and the model reply to the session
            session.add_turn(user_message=input, model_reply=response.text)

        return RunResult(
            final_output=response.text,
            usage=response.usage_metadata,
        )


# ---------------------------------------------------------------
# Define the role and instructions for the AI agent
# ---------------------------------------------------------------
market_researcher_instructions = """
Context:
You are a market research assistant helping analyze companies, industries, and competitors.

Instructions:
When given a question, provide a short factual answer based on your knowledge.

Output:
Start with a verdict prefix: either "✅ FACT:" or "❌ UNKNOWN:"
Follow with a concise one-sentence explanation.
"""

# Create an instance of the Agent
market_researcher_agent = Agent(
    name="Market Researcher",
    instructions=market_researcher_instructions,
    model="gemini-2.5-flash-lite",   # Free-tier Gemini model
)

In [4]:
# ── Example: AI Agent with NO memory ─────────────────────────────

# First question to the AI agent
q1 = "What is the market share of Tesla in the US EV market?"

# Display the user's question
print_markdown(f"You: '{q1}'")

# Run the agent (no session = no memory)
resp1 = Runner.run(starting_agent=market_researcher_agent, input=q1)

# Display the agent's response
print_markdown(f"🤖 Agent:\n{resp1.final_output}")

You: 'What is the market share of Tesla in the US EV market?'

🤖 Agent:
✅ FACT: Tesla held approximately 62% of the US EV market share in the first quarter of 2023.

In [5]:
# Second question — depends on the previous context
# Without memory the agent will NOT remember what "that" refers to
q2 = "How does that compare to last year?"

# Display the follow-up question
print_markdown(f"\nYou: '{q2}'")

# Run the agent again — no memory, so context is lost
resp2 = Runner.run(starting_agent=market_researcher_agent, input=q2)

# Display the agent's response (will fail to connect to the first question)
print_markdown(f"🤖 Agent:\n{resp2.final_output}")


You: 'How does that compare to last year?'

🤖 Agent:
❌ UNKNOWN: I need more information about what "that" refers to and what specific metrics from last year you would like to compare it against.

🛑 As expected, the agent has **no memory**.

# TASK 3: ADDING MEMORY WITH GeminiSession

The OpenAI Agents SDK provides a built-in `SQLiteSession` class.  
Gemini does **not** have an equivalent built-in, but we can implement the same behaviour by:

1. Keeping a **conversation history list** in memory (`GeminiSession` below), or
2. Persisting that list to a **SQLite database** (`SQLiteGeminiSession` below).

Both classes expose the same two-method interface:
- `get_history()` → returns the list of past turns in Gemini's format
- `add_turn(user_message, model_reply)` → appends a new turn

`Runner.run()` picks up the session automatically — no other code changes needed.

In [6]:
import sqlite3
import json

# ---------------------------------------------------------------
# Option A: In-memory session (resets when the kernel restarts)
# ---------------------------------------------------------------
class GeminiSession:
    """
    In-memory conversation history.
    Equivalent to OpenAI Agents SDK SQLiteSession but stores
    history in a plain Python list — no database required.
    """

    def __init__(self):
        # Gemini expects history as a list of dicts:
        # [{"role": "user", "parts": ["..."]}, {"role": "model", "parts": ["..."]}]
        self._history = []

    def get_history(self):
        """Return the full conversation history."""
        return self._history

    def add_turn(self, user_message: str, model_reply: str):
        """Append a user turn and the model's reply."""
        self._history.append({"role": "user",  "parts": [user_message]})
        self._history.append({"role": "model", "parts": [model_reply]})


# ---------------------------------------------------------------
# Option B: SQLite-backed session (persists across kernel restarts)
# Mirrors the behaviour of SQLiteSession from OpenAI Agents SDK.
# ---------------------------------------------------------------
class SQLiteGeminiSession:
    """
    SQLite-backed conversation history.
    Equivalent to `SQLiteSession` from the OpenAI Agents SDK.
    History survives kernel restarts and can be inspected with
    any SQLite browser.
    """

    def __init__(self, session_id: str, db_path: str = "gemini_sessions.db"):
        """
        session_id – unique name for this conversation (like a thread ID)
        db_path    – path to the SQLite database file
        """
        self.session_id = session_id
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        """Create the table if it doesn't already exist."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS history (
                    id         INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT    NOT NULL,
                    role       TEXT    NOT NULL,
                    parts      TEXT    NOT NULL
                )
            """)
            conn.commit()

    def get_history(self):
        """Return the full conversation history for this session."""
        with sqlite3.connect(self.db_path) as conn:
            rows = conn.execute(
                "SELECT role, parts FROM history WHERE session_id = ? ORDER BY id",
                (self.session_id,)
            ).fetchall()
        return [{"role": row[0], "parts": json.loads(row[1])} for row in rows]

    def add_turn(self, user_message: str, model_reply: str):
        """Persist a user turn and the model's reply to the database."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                "INSERT INTO history (session_id, role, parts) VALUES (?, ?, ?)",
                (self.session_id, "user", json.dumps([user_message]))
            )
            conn.execute(
                "INSERT INTO history (session_id, role, parts) VALUES (?, ?, ?)",
                (self.session_id, "model", json.dumps([model_reply]))
            )
            conn.commit()

print("Session classes defined: GeminiSession (in-memory) and SQLiteGeminiSession (persistent).")

Session classes defined: GeminiSession (in-memory) and SQLiteGeminiSession (persistent).


In [7]:
# ── Using SQLiteGeminiSession (equivalent to SQLiteSession in OpenAI SDK) ──
#
# Change to GeminiSession() for a lightweight in-memory alternative.

session = SQLiteGeminiSession("conversation")

# First interaction
user_input1 = "What is the market share of Tesla in the US EV market?"

response1 = Runner.run(
    starting_agent=market_researcher_agent,
    input=user_input1,
    session=session,
)

print_markdown(f"🤖 Agent:\n{response1.final_output}")

🤖 Agent:
✅ FACT: Tesla's market share in the US EV market is estimated to be around 60-70% as of late 2023.

In [8]:
# Second interaction — the agent NOW remembers the previous exchange
user_input2 = "How does that compare to last year?"

response2 = Runner.run(
    starting_agent=market_researcher_agent,
    input=user_input2,
    session=session,
)

print_markdown(f"🤖 Agent:\n{response2.final_output}")

🤖 Agent:
✅ FACT: Tesla's market share in the US EV market has decreased slightly compared to last year, falling from over 70% in 2022 to the current range of 60-70%.

✅ Success! Now the agent **remembers Tesla** because we provided the history ourselves.

**PRACTICE OPPORTUNITY:**  
- **Using the Gemini SDK, create a new AI agent named `Travel Planner` that always suggests one sunny (warm) weekend getaway destination.**  
    - **1. Write the agent's instructions; tell it to only give one sunny/warm destination.**  
    - **2. Use the free `gemini-1.5-flash` model (or `gemini-2.0-flash` for the latest generation).**  
    - **3. Save the conversation history with `SQLiteGeminiSession` so it remembers past user questions and doesn't repeat destinations.**  
    - **4. Ask the agent:**  
      **`Please suggest one weekend destination within 5 hours flying from Toronto, Canada`**  
    - **5. Display the agent's answer.**  
    - **6. Then ask a follow-up question about visa requirements for Canadians for that destination and display the answer.**  
    - **Hint: Use `Runner.run()` to send the message to the agent and print the reply.**  

# PRACTICE OPPORTUNITY SOLUTIONS

In [9]:
# ── 1. Define the Travel Planner agent instructions ─────────────
travel_planner_instructions = """You are an assistant specialising in weekend travel planning.
Always suggest exactly ONE sunny (warm) destination for a quick getaway.
Remember past suggestions and never repeat a destination already mentioned in this conversation.
"""

# ── 2. Create the Agent (free Gemini model) ─────────────────────
travel_planner_agent = Agent(
    name="Travel Planner",
    instructions=travel_planner_instructions,
    model="gemini-2.5-flash-lite",   # or "gemini-2.0-flash" for latest generation
)

In [10]:
# ── 3. Create a persistent session (equivalent to SQLiteSession) ─
session_travel = SQLiteGeminiSession("travel_planner_session")

# ── 4 & 5. First question ────────────────────────────────────────
travel_msg = "Please suggest one weekend destination within 5 hours flying from Toronto, Canada"

resp1_travel = Runner.run(
    starting_agent=travel_planner_agent,
    input=travel_msg,
    session=session_travel,
)

print_markdown(f"Travel Planner:\n{resp1_travel.final_output}")

Travel Planner:
For a sunny weekend getaway from Toronto, I suggest **Charleston, South Carolina**. It's a charming historic city with beautiful architecture, delicious Southern cuisine, and a pleasant climate, offering plenty of sunshine for exploring.

In [11]:
# ── 6. Follow-up question (agent remembers the destination) ──────
followup_travel = "What are the visa requirements for Canadians to this destination?"

resp2_travel = Runner.run(
    starting_agent=travel_planner_agent,
    input=followup_travel,
    session=session_travel,
)

print_markdown(f"Travel Planner:\n{resp2_travel.final_output}")

Travel Planner:
Canadians do not require a visa to visit the United States (which includes Charleston, South Carolina) for tourism or business for stays of up to six months. You will typically need a valid passport.